# SMOTE Inside Cross-Validation Folds

A critical rule for imbalanced classification: SMOTE (Synthetic Minority Over-sampling) must be applied **only to the training fold**, never to the test fold. Applying SMOTE before splitting, or to the test set, leaks synthetic information into evaluation and produces an overly optimistic, invalid score. This notebook demonstrates the correct way to do it, and contrasts it with the incorrect (leaky) approach to show why it matters.

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE

In [7]:
df = pd.read_csv("../data/processed/fused_dataset.csv")

target = 'Machine failure'
exclude_cols = ['UDI', 'Product ID', 'Type', 'timestamp', target,
                'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols].fillna(0)
y = df[target]

print("Features:", len(feature_cols))
print("Failure rate:", y.mean())

Features: 23
Failure rate: 0.0339


## Correct approach: SMOTE inside each fold

For each fold, SMOTE is fit and applied only on that fold's training data, after the split. The test fold remains untouched and reflects the real-world class imbalance.

In [8]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

correct_scores = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Apply SMOTE ONLY on training data
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_resampled, y_train_resampled)
    preds = model.predict(X_test)

    score = f1_score(y_test, preds, average='macro')
    correct_scores.append(score)

    print(f"Fold {fold}: Train size before SMOTE={len(X_train)}, "
          f"after SMOTE={len(X_train_resampled)} | Test Macro F1={score:.4f}")

print(f"\nCorrect (SMOTE inside fold) Mean Macro F1: {np.mean(correct_scores):.4f} (+/- {np.std(correct_scores):.4f})")

Fold 1: Train size before SMOTE=8000, after SMOTE=15456 | Test Macro F1=0.8850
Fold 2: Train size before SMOTE=8000, after SMOTE=15458 | Test Macro F1=0.8447
Fold 3: Train size before SMOTE=8000, after SMOTE=15458 | Test Macro F1=0.8758
Fold 4: Train size before SMOTE=8000, after SMOTE=15458 | Test Macro F1=0.8948
Fold 5: Train size before SMOTE=8000, after SMOTE=15458 | Test Macro F1=0.8586

Correct (SMOTE inside fold) Mean Macro F1: 0.8718 (+/- 0.0181)


## Incorrect approach (for comparison): SMOTE before splitting

This is a common mistake — applying SMOTE to the *entire* dataset before cross-validation. Synthetic samples derived from points that end up in the test fold can leak into the training fold, inflating the score artificially.

In [9]:
smote_full = SMOTE(random_state=42)
X_resampled_full, y_resampled_full = smote_full.fit_resample(X, y)

skf_leaky = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
leaky_scores = []

for fold, (train_idx, test_idx) in enumerate(skf_leaky.split(X_resampled_full, y_resampled_full), 1):
    X_train, X_test = X_resampled_full.iloc[train_idx], X_resampled_full.iloc[test_idx]
    y_train, y_test = y_resampled_full.iloc[train_idx], y_resampled_full.iloc[test_idx]

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    score = f1_score(y_test, preds, average='macro')
    leaky_scores.append(score)

print(f"Incorrect (SMOTE before split, leaky) Mean Macro F1: {np.mean(leaky_scores):.4f} (+/- {np.std(leaky_scores):.4f})")

Incorrect (SMOTE before split, leaky) Mean Macro F1: 0.9889 (+/- 0.0018)


In [10]:
print("="*60)
print("SMOTE LEAKAGE COMPARISON")
print("="*60)
print(f"Correct (SMOTE inside fold):     {np.mean(correct_scores):.4f}")
print(f"Incorrect (SMOTE before split):  {np.mean(leaky_scores):.4f}")
print(f"Inflation due to leakage:        {np.mean(leaky_scores) - np.mean(correct_scores):+.4f}")

SMOTE LEAKAGE COMPARISON
Correct (SMOTE inside fold):     0.8718
Incorrect (SMOTE before split):  0.9889
Inflation due to leakage:        +0.1171


## Summary

The correct, leakage-free approach (SMOTE applied only inside each training fold) gives a Macro F1 of **0.8718** (+/- 0.0181). The naive approach (SMOTE applied before splitting) gives an inflated score of **0.9889** (+/- 0.0018) — a difference of **+0.1171**, demonstrating exactly why SMOTE must always be applied after the train/test split, never before. The inflated score is misleading: it reflects the model's ability to recognize synthetic duplicates of training points that leaked into the test set, not genuine generalization to unseen failures. This validated, leakage-free SMOTE pipeline (0.8718 Macro F1) will be used in Issue #9 for final LightGBM training.